# 01 · Getting started with Scrapy.Net

Fetch a page, try selectors interactively, then write and run a first spider against quotes.toscrape.com.

*Ambil halaman, coba selector secara interaktif, lalu tulis dan jalankan spider pertama.*

*Dibuat oleh Gravicode Studios, dipimpin oleh Kang Fadhil.*

In [ ]:
// Packages / Paket
#r "nuget: Gravicode.ScrapyNet, 0.1.0"

// Before the packages are on nuget.org, point at a local build instead:
//   dotnet pack ScrapyNet.slnx -c Release -o artifacts
// and add this line above the #r lines:  #i "nuget: ../artifacts"

## Fetch one page

`Fetcher` downloads through the full engine (headers, cookies, retries), like `scrapy shell`.

*`Fetcher` mengunduh lewat mesin lengkap, seperti `scrapy shell`.*

In [ ]:
using ScrapyNet;
using ScrapyNet.Cmdline;

var response = await Fetcher.FetchAsync("https://quotes.toscrape.com/");
display($"{response.Status} {response.Url} ({response.Body.Length:N0} bytes)");

## Selectors

CSS with Scrapy's `::text` and `::attr()`, XPath, and regular expressions.

*CSS dengan `::text` dan `::attr()`, XPath, serta regex.*

In [ ]:
display(response.Css("title::text").Get());
display(response.Css("small.author::text").GetAll().Distinct());
display(response.Xpath("//div[@class='quote'][1]//a[@class='tag']/text()").GetAll());
display(response.Css("span.text::text").ReFirst(@"“(\w+)"));

## A first spider

Callbacks yield items (any object) and follow-up requests.

*Callback menghasilkan item (objek apa pun) dan request lanjutan.*

In [ ]:
public sealed class QuotesSpider : Spider
{
    public override string Name => "quotes";
    public override IReadOnlyList<string> StartUrls => ["https://quotes.toscrape.com/"];

    public override async IAsyncEnumerable<object> Parse(Response response)
    {
        foreach (var quote in response.Css("div.quote"))
            yield return new
            {
                Text = quote.Css("span.text::text").Get(),
                Author = quote.Css("small.author::text").Get(),
                Tags = quote.Css("a.tag::text").GetAll(),
            };

        if (response.Css("li.next a::attr(href)").Get() is { } next)
            yield return response.Follow(next, Parse);
        await Task.CompletedTask;
    }
}

In [ ]:
var settings = new Settings();
settings.Set(SettingKeys.LogLevel, "WARNING");
settings.Set(SettingKeys.CloseSpiderPageCount, 2);   // be polite: two pages are enough here
settings.Set(SettingKeys.DownloadDelay, 0.5);

var (result, items) = await Scrapy.CrawlAndCollectAsync(new QuotesSpider(), settings);
display(result);
display(items.Take(3).Select(i => ItemAdapter.For(i).ToString()));